In [3]:
import os
import requests
import csv
from datetime import datetime
from time import sleep
from fhir_fetcher import fetch_all_data  # Ensure this module is available

# and handles paging through all records


def fetch_patient_observations(session, fhir_base_url, patient_id):
    qstr = f"Observation?subject=Patient/{patient_id}"
    start_url = f"{fhir_base_url}/{qstr}"
    observations = fetch_all_data(
        session, start_url, 0
    )  # Fetch all observations for the patient
    return observations


def extract_observation_data(observations):
    data = {}
    for entry in observations:
        resource = entry.get("resource", {})
        code = resource.get("code", {}).get("coding", [{}])[0]
        attribute_name = code.get("display", "")
        value_string = resource.get("valueString", "")
        value_quantity = resource.get("valueQuantity", {}).get("value", "")
        if attribute_name:
            value = value_string if value_string else value_quantity
            if value:
                data[attribute_name] = value
    return data


def fetch_patient_ids(session, fhir_base_url, study_reference):
    query_url = f"{fhir_base_url}/ResearchSubject?study={study_reference}"
    print(query_url)
    research_subjects = fetch_all_data(session, query_url, 0, "n")
    patient_ids = [
        entry["resource"]["individual"]["reference"].split("/")[-1]
        for entry in research_subjects
    ]
    return patient_ids


def main():
    starttime = datetime.now()
    starttimeStr = starttime.strftime("%Y-%m-%d %H:%M:%S")
    print("====== start time:", starttimeStr)

    ###############################################################################################
    # get the token from https://www.ncbi.nlm.nih.gov/gap/power-user-portal/.
    #  Scroll down and click on the "Task Specific Token" button to get the light-weight version of the dbGaP RAS Passport.
    #  Save the file into a text. In my example, it is saved to task-specific-token_all.txt.
    ###############################################################################################
    TST_PATH = "~/dev/fhir/task-specific-token-all.txt"
    fhir_base_url = "https://dbgap-api.ncbi.nlm.nih.gov/fhir-jpa-pilot1/x1"

    with open(os.path.expanduser(TST_PATH), "r") as f:
        tst_token = f.read().strip()

    session = requests.Session()
    session.headers.update(
        {
            "Accept": "application/fhir+json",
            "Authorization": f"Bearer {tst_token}",
            "Content-Type": "application/x-www-form-urlencoded",
        }
    )

    # study_reference = "phs002921.v2.p1.c1"
    study_reference = "phs002921"

    ################################################################################################
    # https://dbgap-api.ncbi.nlm.nih.gov/fhir-jpa-pilot/x1/ResearchSubject?study=phs002921
    # https://dbgap-api.ncbi.nlm.nih.gov/fhir-jpa-pilot/x1/Observation?subject=Patient/4317770
    # ###############################################################################################

    patient_ids = fetch_patient_ids(session, fhir_base_url, study_reference)
    print(f"Total patients fetched: {len(patient_ids)}")

    data = []
    columns = set()
    patients_with_observations = 0
    for patient_id in patient_ids[:2]:
        observations = fetch_patient_observations(
            session, fhir_base_url, patient_id
        )
        observation_data = extract_observation_data(observations)
        if observation_data:
            observation_data["Patient"] = patient_id
            columns.update(observation_data.keys())
            data.append(observation_data)
            patients_with_observations += 1
            # print(f"Observations obtained for patient: {patient_id}")
            print(
                f"Accumulative patients with observations: {patients_with_observations}"
            )

        sleep(
            1
        )  # Add a delay of 1 second between each patient API request to avoid rate limits

    columns = ["Patient"] + sorted(
        columns
    )  # Ensure 'Patient' is the first column

    output_file = "patient_observations.csv"
    with open(output_file, "w", newline="") as csvfile:
        csvwriter = csv.DictWriter(csvfile, fieldnames=columns)
        csvwriter.writeheader()
        csvwriter.writerows(data)

    print(f"Data written to {output_file}")

    endtime = datetime.now()
    endtimeStr = endtime.strftime("%Y-%m-%d %H:%M:%S")
    print("====== end time:", endtimeStr)

    elapsed_time = endtime - starttime
    elapsed_seconds = elapsed_time.total_seconds()
    eminutes = elapsed_seconds // 60
    eseconds = elapsed_seconds % 60

    print(
        f"===========Elapsed time: {int(eminutes)} minutes and {int(eseconds)} seconds."
    )


if __name__ == "__main__":
    main()

====== start time: 2024-08-07 15:31:34
https://dbgap-api.ncbi.nlm.nih.gov/fhir-jpa-pilot1/x1/ResearchSubject?study=phs002921
Total patients fetched: 1035
Accumulative patients with observations: 1
Accumulative patients with observations: 2
Data written to patient_observations.csv
====== end time: 2024-08-07 15:32:03
===========Elapsed time: 0 minutes and 28 seconds.
